# Data Prep to create Silver Parquet

## Prerequisites

data_loader -> bronze_chicago_taxi_2024.parquet

In [5]:
import pandas as pd
import requests
from pathlib import Path
import duckdb
import polars as pl
import numpy as np

TAXI_PATH = "bronze_chicago_taxi_2024.parquet"
WEATHER_PATH = "bronze_weatherdata.parquet"

# Taxi Data

## Null Values

column_name                 n_rows   n_nulls  null_pct
      dropoff_census_tract 15406960 8751654.0     56.80
       pickup_census_tract 15406960 8547337.0     55.48
    dropoff_community_area 15406960 1366538.0      8.87
 dropoff_centroid_location 15406960 1285413.0      8.34
 dropoff_centroid_latitude 15406960 1285413.0      8.34
dropoff_centroid_longitude 15406960 1285413.0      8.34
     pickup_community_area 15406960  429216.0      2.79
  pickup_centroid_latitude 15406960  421354.0      2.73
 pickup_centroid_longitude 15406960  421354.0      2.73 
  pickup_centroid_location 15406960  421354.0      2.73 -> 
                      fare 15406960   31816.0      0.21 -> Estimate, average fare for "trip_seconds" or more precise some range of it
                      tips 15406960   31816.0      0.21 -> Estimate, average tips for "trip_seconds" or more precise some range of it
                     tolls 15406960   31816.0      0.21 -> Estimate, average tolls for "trip_seconds" or more precise some range of it
                    extras 15406960   31816.0      0.21 -> Estimate, average extras for "trip_seconds" or more precise some range of it
                trip_total 15406960   31816.0      0.21 -> Estimate, sum of fare, tips, tolls, extras
              trip_seconds 15406960    2936.0      0.02 -> Estimate, "trip_end_timestamp" - "trip_start_timestamp" (if both are not null)
                   trip_id 15406960       0.0      0.00 -> Ok
                   taxi_id 15406960       0.0      0.00 -> Ok
      trip_start_timestamp 15406960       0.0      0.00 -> Ok
        trip_end_timestamp 15406960     345.0      0.00 -> Estimate, "trip_start_timestamp" + "trip_seconds" (if "trip_seconds" not null)
                trip_miles 15406960     130.0      0.00 -> Estimate, average trip_miles for "trip_seconds" or more precise some range of it
               :created_at 15406960       0.0      0.00 -> Ok, drop anyways
               :updated_at 15406960       0.0      0.00 -> Ok, drop anyways
                  :version 15406960       0.0      0.00 -> Ok, drop anyways
              payment_type 15406960       0.0      0.00 -> Ok
                   company 15406960       0.0      0.00 -> Ok
                       :id 15406960       0.0      0.00 -> Ok, drop anyways

# Weather Data

## Null Values
column_name  n_rows  n_nulls  null_pct
       gust   24360  18401.0     75.54 -> drop, too many missing values and likely no value add as wind speed is available
      skyc3   24360  18227.0     74.82 -> drop, too many missing values and likely no value add as cloud cover level 1 is available
      skyc2   24360  11411.0     46.84 -> drop, too many missing values and likely no value add as cloud cover level 1 is available
       p01m   24360      3.0      0.01 -> interpolate
       relh   24360      2.0      0.01 -> interpolate
       tmpc   24360      2.0      0.01 -> interpolate
    station   24360      0.0      0.00 -> Ok
      skyc1   24360      0.0      0.00 -> Ok
        lon   24360      0.0      0.00 -> Ok
      valid   24360      0.0      0.00 -> Ok
       vsby   24360      0.0      0.00 -> Ok
       sknt   24360      1.0      0.00 -> interpolate
        lat   24360      0.0      0.00 -> Ok

In [4]:
df = pd.read_parquet("bronze_weatherdata.parquet")

# 1. String-"null" und ähnliche Platzhalter in echte Missing Values umwandeln
null_like_values = ["null", "NULL", "Null", "", "nan", "NaN", "None", "none", "NA", "N/A", "n/a"]

df = df.replace(null_like_values, pd.NA)

# 2. Spalten mit zu vielen Missing Values droppen
drop_cols = ["gust", "skyc3", "skyc2"]
existing_drop_cols = [col for col in drop_cols if col in df.columns]

df = df.drop(columns=existing_drop_cols)

# 3. Timestamp parsen
if "valid" in df.columns:
    df["valid"] = pd.to_datetime(df["valid"], errors="coerce")
    df = df.sort_values("valid").reset_index(drop=True)

# 4. Numerische Spalten casten
interpolate_cols = ["p01m", "relh", "tmpc", "sknt"]
existing_interpolate_cols = [col for col in interpolate_cols if col in df.columns]

for col in existing_interpolate_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# 5. Fehlende Werte interpolieren
for col in existing_interpolate_cols:
    df[col] = df[col].interpolate(method="linear")

# 6. Falls Missing Values am Anfang/Ende liegen
for col in existing_interpolate_cols:
    df[col] = df[col].ffill().bfill()
    
df.to_parquet(
    "silver_weatherdata.parquet",
    index=False,
    engine="pyarrow",
    compression="snappy",
)

# Data Prep to create Gold Parquet

## Taxi Data

## Weather Data

Create hourly indexed dataset for weather data from 01.01.2024 to 25.05.2026

In [6]:
SILVER_WEATHER_PATH = Path("silver_weatherdata.parquet")
GOLD_WEATHER_PATH = Path("gold_weather_hourly.parquet")

START_TS = "2024-01-01 00:00:00"
END_TS = "2026-05-25 23:00:00"


def mode_or_na(series: pd.Series):
    """Return most frequent non-null value, otherwise NA."""
    mode_values = series.dropna().mode()
    if len(mode_values) == 0:
        return pd.NA
    return mode_values.iloc[0]


def create_hourly_weather_gold(
    df: pd.DataFrame,
    start_ts: str = START_TS,
    end_ts: str = END_TS,
) -> pd.DataFrame:
    """
    Create hourly gold weather dataset.

    Steps:
    - Parse valid timestamp
    - Restrict to requested date range
    - Aggregate observations to hourly level
    - Create complete hourly timestamp spine
    - Reindex to all hours
    - Linearly interpolate numeric weather columns
    - Fill categorical/context columns
    - Add date/hour helper columns
    """

    df = df.copy()

    # 1. Basic cleanup
    df["valid"] = pd.to_datetime(df["valid"], errors="coerce")
    df = df.dropna(subset=["valid"])

    start_ts = pd.Timestamp(start_ts)
    end_ts = pd.Timestamp(end_ts)

    # Optional: keep only relevant date range
    df = df[(df["valid"] >= start_ts) & (df["valid"] <= end_ts)]

    # 2. Create hourly timestamp
    df["valid_hour"] = df["valid"].dt.floor("h")

    # 3. Cast numeric columns
    numeric_cols = [
        "tmpc",   # temperature Celsius
        "relh",   # relative humidity
        "sknt",   # wind speed in knots
        "p01m",   # precipitation
        "vsby",   # visibility
        "lat",
        "lon",
    ]

    existing_numeric_cols = [col for col in numeric_cols if col in df.columns]

    for col in existing_numeric_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    # 4. Define aggregation rules
    agg_dict = {}

    # Numeric weather values: hourly mean
    for col in ["tmpc", "relh", "sknt", "vsby"]:
        if col in df.columns:
            agg_dict[col] = "mean"

    # Precipitation: hourly sum is usually more sensible than mean
    if "p01m" in df.columns:
        agg_dict["p01m"] = "sum"

    # Station/location fields
    if "station" in df.columns:
        agg_dict["station"] = mode_or_na

    if "lat" in df.columns:
        agg_dict["lat"] = "mean"

    if "lon" in df.columns:
        agg_dict["lon"] = "mean"

    # Categorical weather condition
    if "skyc1" in df.columns:
        agg_dict["skyc1"] = mode_or_na

    # 5. Aggregate to hourly level
    hourly = (
        df
        .groupby("valid_hour", as_index=True)
        .agg(agg_dict)
        .sort_index()
    )

    # 6. Complete hourly index from start to end
    full_hourly_index = pd.date_range(
        start=start_ts,
        end=end_ts,
        freq="h",
        name="valid_hour",
    )

    hourly = hourly.reindex(full_hourly_index)

    # 7. Interpolate numeric columns over missing hours
    numeric_interpolate_cols = [
        col for col in ["tmpc", "relh", "sknt", "p01m", "vsby", "lat", "lon"]
        if col in hourly.columns
    ]

    hourly[numeric_interpolate_cols] = (
        hourly[numeric_interpolate_cols]
        .interpolate(method="time", limit_direction="both")
    )

    # 8. Fill categorical/context columns
    categorical_fill_cols = [
        col for col in ["station", "skyc1"]
        if col in hourly.columns
    ]

    for col in categorical_fill_cols:
        hourly[col] = hourly[col].ffill().bfill()

    # 9. Back to normal dataframe
    gold = hourly.reset_index()

    # 10. Add helper columns for joins / ML
    gold["date"] = gold["valid_hour"].dt.date
    gold["year"] = gold["valid_hour"].dt.year
    gold["month"] = gold["valid_hour"].dt.month
    gold["day"] = gold["valid_hour"].dt.day
    gold["hour"] = gold["valid_hour"].dt.hour
    gold["weekday"] = gold["valid_hour"].dt.weekday

    # 11. Optional quality flags
    original_hours = set(df["valid_hour"].dropna().unique())

    gold["was_observed_hour"] = gold["valid_hour"].isin(original_hours)
    gold["was_interpolated_hour"] = ~gold["was_observed_hour"]

    return gold


GOLD_WEATHER_PATH.parent.mkdir(parents=True, exist_ok=True)

weather_silver = pd.read_parquet(SILVER_WEATHER_PATH)

weather_gold = create_hourly_weather_gold(
    weather_silver,
    start_ts=START_TS,
    end_ts=END_TS,
)

weather_gold.to_parquet(
    GOLD_WEATHER_PATH,
    index=False,
    engine="pyarrow",
    compression="snappy",
)

print(f"Gold weather parquet written to: {GOLD_WEATHER_PATH}")
print(f"Rows: {len(weather_gold):,}")
print(f"Start: {weather_gold['valid_hour'].min()}")
print(f"End: {weather_gold['valid_hour'].max()}")
print()
print(weather_gold.head().to_string(index=False))
print()
print(weather_gold.tail().to_string(index=False))

Gold weather parquet written to: gold_weather_hourly.parquet
Rows: 21,024
Start: 2024-01-01 00:00:00
End: 2026-05-25 23:00:00

         valid_hour  tmpc  relh  sknt  vsby  p01m station    lat      lon skyc1       date  year  month  day  hour  weekday  was_observed_hour  was_interpolated_hour
2024-01-01 00:00:00  1.11 78.43  13.0  10.0   0.0     MDW 41.786 -87.7524   OVC 2024-01-01  2024      1    1     0        0               True                  False
2024-01-01 01:00:00  1.11 78.43  14.0  10.0   0.0     MDW 41.786 -87.7524   OVC 2024-01-01  2024      1    1     1        0               True                  False
2024-01-01 02:00:00  1.11 78.43  14.0  10.0   0.0     MDW 41.786 -87.7524   OVC 2024-01-01  2024      1    1     2        0               True                  False
2024-01-01 03:00:00  1.11 78.43  16.0  10.0   0.0     MDW 41.786 -87.7524   BKN 2024-01-01  2024      1    1     3        0               True                  False
2024-01-01 04:00:00  1.11 78.43  14.0  10.0